# T5-base Inference on Spider Dev

Loads the fine-tuned checkpoint from Drive, generates SQL for all 1034 dev examples, writes `pred_t5.txt` and `gold.txt` aligned to `dev.json`.

Runtime: ~5-10 min on Colab T4. Files are auto-downloaded at the end.

**Must use the exact same schema serialization as during training** — don't edit `build_schema_string` or `make_input`.

In [ ]:
!pip install -q "transformers>=4.40"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/text-to-sql')
SPIDER_DIR = DRIVE_ROOT / 'spider_data'
CHECKPOINT = DRIVE_ROOT / 't5_base_spider' / 'final'
OUT_DIR = Path('/content/predictions')
OUT_DIR.mkdir(exist_ok=True)

assert CHECKPOINT.exists(), f'No checkpoint at {CHECKPOINT}. Run fine-tuning notebook first.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)

validation = load_json(SPIDER_DIR / 'dev.json')
tables = load_json(SPIDER_DIR / 'tables.json')
print(f'dev examples: {len(validation)}')

In [ ]:
def build_schema_string(db):
    table_names = db['table_names_original']
    columns = db['column_names_original']
    column_types = db['column_types']
    foreign_keys = db.get('foreign_keys', [])

    tables_to_cols = {i: [] for i in range(len(table_names))}
    col_idx_to_ref = {}
    for col_idx, (tbl_idx, col_name) in enumerate(columns):
        if tbl_idx == -1:
            continue
        col_type = column_types[col_idx]
        tables_to_cols[tbl_idx].append(f'{col_name}:{col_type}')
        col_idx_to_ref[col_idx] = f'{table_names[tbl_idx]}.{col_name}'

    table_parts = []
    for i, tbl in enumerate(table_names):
        cols = ', '.join(tables_to_cols[i])
        table_parts.append(f'{tbl}({cols})')

    fk_parts = []
    for a, b in foreign_keys:
        if a in col_idx_to_ref and b in col_idx_to_ref:
            fk_parts.append(f'{col_idx_to_ref[a]} = {col_idx_to_ref[b]}')

    schema = ' | '.join(table_parts)
    if fk_parts:
        schema += ' | foreign_keys: ' + ', '.join(fk_parts)
    return schema

schema_by_db = {t['db_id']: build_schema_string(t) for t in tables}

def make_input(ex):
    return f"translate English to SQL | {ex['db_id']} | {ex['question']} | schema: {schema_by_db[ex['db_id']]}"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT))
model = AutoModelForSeq2SeqLM.from_pretrained(str(CHECKPOINT)).to(device)
model.eval()
print('Model loaded')

In [ ]:
with open(OUT_DIR / 'gold.txt', 'w', encoding='utf-8') as f:
    for ex in validation:
        q = ' '.join(ex['query'].split())
        f.write(f"{q}\t{ex['db_id']}\n")
print('gold.txt written')

In [ ]:
BATCH_SIZE = 16
MAX_INPUT_LEN = 512
MAX_NEW_TOKENS = 256
NUM_BEAMS = 4

preds = []
with torch.no_grad():
    for start in tqdm(range(0, len(validation), BATCH_SIZE)):
        batch = validation[start:start + BATCH_SIZE]
        inputs_text = [make_input(ex) for ex in batch]
        enc = tokenizer(
            inputs_text,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(device)
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

print(f'Generated {len(preds)} predictions')

In [ ]:
with open(OUT_DIR / 'pred_t5.txt', 'w', encoding='utf-8') as f:
    for p in preds:
        p = ' '.join(p.split()).strip()
        if not p:
            p = 'SELECT 1'
        f.write(p + '\n')

n_gold = sum(1 for _ in open(OUT_DIR / 'gold.txt'))
n_pred = sum(1 for _ in open(OUT_DIR / 'pred_t5.txt'))
print(f'gold.txt: {n_gold} lines')
print(f'pred_t5.txt: {n_pred} lines')
assert n_gold == n_pred, 'Alignment broken'

In [ ]:
for i in range(5):
    print(f'--- example {i} ---')
    print('Q   :', validation[i]['question'])
    print('GOLD:', validation[i]['query'])
    print('PRED:', preds[i])
    print()

In [ ]:
from google.colab import files
files.download(str(OUT_DIR / 'pred_t5.txt'))
files.download(str(OUT_DIR / 'gold.txt'))